# Next step: step 7a, unperturbed galaxies
**What it adds:** `sim.unperturbed_particles(params, config, seed)`, the two galaxies as they'd look now if they had
never interacted.

| | What it is |
|---|---|
| `sim.unperturbed_particles(params, config, seed)` | The same disks a run starts with (same seed), placed around today's centers with no steps. Same arguments and shape as `simulate_particles` |
| `jspam.initial_state(..., t_start=0)` | What it calls: the setup only, at now. 0 is allowed here only |

**Why:** the image stage needs an unperturbed image beside the perturbed one. Before, that meant taking the starting
disks and moving them to today's centers in the image code. Now the simulator returns them already in place, in
the same coordinates as a run, so the image code treats both the same way.

**What was dropped:** step 7 (many endings, particles at several times in one run). It was for training a deep CNN
that turned out to learn something not wanted, and nothing else in the plan needs it. `sim-branch`'s
`run_particles` (`end_steps`) is the reference if a use appears.

**How it connects:**
- **Below, in JSPAM:** placement steps the secondary back from now while `t_start < t`. At `t_start = 0` that's zero
  steps, so the secondary stays exactly where the model puts it. No Fortran change was needed, only the Python check.
- **`sim`:** `_jspam_args` takes an optional `t_start`; `unperturbed_particles` passes 0, so `_choose_start` (and
  its search) never runs.
- **Above:** the image stage renders `simulate_particles` and `unperturbed_particles` the same way.

The full, current demo is `simulator_tour.ipynb`.

**To run it** (from this repo's root, in bash or PowerShell), then open the `127.0.0.1` link it prints:
```bash
docker run --rm -it -p 8888:8888 -v "${PWD}:/gtc" -w /gtc -e UV_PROJECT_ENVIRONMENT=/tmp/venv \
    ogdenm12/gtc-base-image:v2 uv run --group notebook jupyter lab --ip=0.0.0.0 --allow-root --no-browser
```
- `-v "${PWD}:/gtc"` mounts this repo into the container, so edits on Windows show up immediately.
- `UV_PROJECT_ENVIRONMENT=/tmp/venv` keeps the Linux virtual environment inside the container, out of the repo.
  It's rebuilt each time the container starts (a minute or two, mostly compiling SPAM).
- Our package is installed in editable mode. After editing a `.py` file, restart the kernel to pick it up.

## What changed
**JSPAM, `src/jspam/__init__.py`:** `initial_state` accepts `t_start = 0`; every other function still needs a
negative start (a run starting now would take only a step or two):
```python
def _set_start(t_start, allow_now=False):
    ...
    if not (t_start <= 0 if allow_now else t_start < 0):
        raise ValueError(...)

def initial_state(..., t_start=None):
    ...
    _set_start(t_start, allow_now=True)
```
Why 0 works in the Fortran as it is (`setup_module.f90`, `perturber_position_vec`):
```fortran
tcurrent = 0
do while (t0 < tcurrent)       ! t0 = 0: no steps back
  call wrap_rk41(xx0, h, mass1, mass2, eps1, eps2, xxe)
  ...
enddo
...
x0(i,:) = x0(i,:) + xx0(:)     ! the secondary's disk, moved to its center (now: the model's position)
```

**`sim.py`:**
```python
def unperturbed_particles(params, config=SimulatorConfig(), seed=None):
    return jspam.initial_state(_spam_params(params), seed=seed, **_jspam_args(params, config, t_start=0.0))

def _jspam_args(params, config, t_start=None):      # t_start given: don't choose one
    return {"t_start": _choose_start(params, config)[0] if t_start is None else t_start, ...}
```

**Tests:** the unperturbed disks are the starting disks, with the secondary's moved to the model's position; the
start settings don't change them; a run's secondary ends close to the same center. In JSPAM: the same at the
`jspam` level, and a run (or any positive start) still rejects 0.

In [ ]:
%matplotlib inline
import time
from dataclasses import replace

import jspam
import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import sim

MODEL_STRING = "-9.93853,-4.5805,15.43348,-1.54025,-2.97899,4.69234,34.41671,29.49607,2.97913,4.2501,301.0,35.5,310.80745,321.98757,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0.24008,-2.41999,5.62715,0.0,1.0,0.0,1.0,0.0,0.0"
params = sim.parse_model_string(MODEL_STRING)
N = 2000
config = sim.SimulatorConfig(n1=N, n2=N)
np.set_printoptions(suppress=True)

## 1. The unperturbed galaxies
The last row is the secondary's center: exactly the model's x, y, z, vx, vy, vz, because nothing was stepped. The
primary stays at the origin.

In [ ]:
unperturbed = sim.unperturbed_particles(params, config, seed=1)
print("shape:", unperturbed.shape)
print("secondary's center:", unperturbed[-1])
print("model's x..vz:     ", params[:6])
print("equal:", np.array_equal(unperturbed[-1], params[:6]))

## 2. The same disks a run starts with
`initial_state` (same seed) is the start of a run, before the encounter. The primary's disk is identical; the
secondary's is the same draw, moved with its center from the start position to today's:

In [ ]:
start = sim.initial_state(params, config, seed=1)
print("start time:", sim.get_timing(params, config)["t_start"])
print("primary's disk identical:", np.array_equal(unperturbed[:N], start[:N]))
moved = (unperturbed[N:2 * N] - unperturbed[-1]) - (start[N:2 * N] - start[-1])
print(f"secondary's disk, relative to its center, largest difference: {np.abs(moved).max():.1e}")
print("secondary's center moved by:", (unperturbed[-1, :3] - start[-1, :3]).round(2))

## 3. Unperturbed beside the run
Same seed, same axes, no translating: both come in the coordinates of now.

In [ ]:
run = sim.simulate_particles(params, config, seed=1)
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharex=True, sharey=True)
for ax, (title, state) in zip(axes, [("unperturbed", unperturbed), ("run (perturbed)", run)]):
    ax.scatter(state[:N, 0], state[:N, 1], s=1, label="primary disk")
    ax.scatter(state[N:2 * N, 0], state[N:2 * N, 1], s=1, label="secondary disk")
    ax.plot(0, 0, "k+", ms=12, label="primary center")
    ax.plot(state[-1, 0], state[-1, 1], "kx", ms=12, label="secondary center")
    ax.set_title(title)
    ax.set_aspect("equal")
axes[0].legend(markerscale=5)
plt.show()
print("secondary's center, run:        ", run[-1, :3].round(3))
print("secondary's center, unperturbed:", unperturbed[-1, :3].round(3))

The centers differ slightly: a run ends 1 to 2 steps (0.001 each) past now, so its secondary has moved a little
further. For an image that's far below a pixel.

## 4. What changes the unperturbed galaxies
The start settings don't matter (nothing is stepped). Heat and the mass model set the starting **velocities**, but
the **positions** depend only on the 14 parameters and the seed. Images only use positions.

In [ ]:
changes = {
    "start=-8": {"start": -8.0},
    "start='wallin'": {"start": "wallin"},
    "heat 0.5": {"heat1": 0.5, "heat2": 0.5},
    "r_scale 9": {"r_scale": 9.0},
    "m_halo 11.6": {"m_halo": 11.6},
    "seed 2": None,
}
print(f"{'':16s} {'positions same':>14s} {'largest velocity change':>24s}")
for label, change in changes.items():
    other = (sim.unperturbed_particles(params, config, seed=2) if change is None
             else sim.unperturbed_particles(params, replace(config, **change), seed=1))
    same = np.array_equal(other[:, :3], unperturbed[:, :3])
    print(f"{label:16s} {str(same):>14s} {np.abs(other[:, 3:] - unperturbed[:, 3:]).max():24.4f}")

## 5. Cost
No start-time search and no steps, so it's cheaper than `initial_state` (which searches for the start) and far
cheaper than a run:

In [ ]:
for label, call in [("unperturbed_particles", lambda: sim.unperturbed_particles(params, config, seed=1)),
                    ("initial_state", lambda: sim.initial_state(params, config, seed=1)),
                    ("simulate_particles", lambda: sim.simulate_particles(params, config, seed=1))]:
    start_clock = time.perf_counter()
    call()
    print(f"{label:22s} {time.perf_counter() - start_clock:6.3f} s")

## 6. A start of 0 elsewhere is still an error
Only the setup accepts `t_start = 0`. A run starting now would take a step or two and return almost the starting
disks, which is easy to mistake for a real run, so `jspam` rejects it. `SimulatorConfig(start=0)` is rejected too.

In [ ]:
spam_params = np.concatenate([params, sim.UNUSED_PARAMS])
for label, call in [("jspam.run_collision(t_start=0)", lambda: jspam.run_collision(spam_params, n1=5, n2=5, t_start=0.0)),
                    ("jspam.initial_state(t_start=0.5)", lambda: jspam.initial_state(spam_params, n1=5, n2=5, t_start=0.5)),
                    ("SimulatorConfig(start=0)", lambda: sim.SimulatorConfig(start=0))]:
    try:
        call()
        print(f"{label} -> no error")
    except Exception as error:  # print a failure instead of stopping "Run All"
        print(f"{label} -> {type(error).__name__}: {error}")